# Pump Hydraulic Model — Colebrook Verification

**Purpose:** independently assess the constant-friction-factor approximation used in the original pump-system model.

This notebook:
1. Reconstructs the original pump and constant-friction system curves.
2. Calculates a variable Darcy friction factor using the Colebrook–White equation.
3. Recalculates the operating point using the variable-friction model.
4. Quantifies the difference between the two models.
5. Exports a flow-sweep dataset and comparison figures.

**Scope and assumptions**
- Water properties are fixed at 25 °C.
- Pipe diameter, length and roughness are treated as constant.
- The hydraulic model includes straight-pipe friction and the 8 m static head. Minor losses are not included here.
- The Colebrook calculation is used over the turbulent flow range considered. The transition region is not modelled because it is outside the selected 10–96 m³/h comparison range.
- The pump curve is reconstructed from the supplied manufacturer data using a third-order polynomial fit.


## 1. Imports and original model parameters

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import brentq

# Original pump-system model
STATIC_HEAD = 8.0                 # m
K_ORIGINAL = 0.001093466742       # m / (m3/h)^2

# Pipe and fluid parameters
D = 0.100                         # internal diameter, m
L = 95.0                          # total straight-pipe length, m
EPSILON = 0.0000457               # absolute roughness, m
RHO = 997.0                       # water density at 25 °C, kg/m3
MU = 0.000891                     # dynamic viscosity at 25 °C, Pa.s
G = 9.81                          # gravitational acceleration, m/s2

print("Hydraulic analysis environment ready.")


## 2. Manufacturer pump curve

In [ ]:
flow_data = np.array([
    0, 8, 16, 24, 32, 40, 48,
    56, 64, 72, 80, 88, 96
], dtype=float)

head_data = np.array([
    15.6, 15.55306048, 15.52544384,
    15.48655296, 15.40579072,
    15.25256, 14.99626368,
    14.60630464, 14.05208576,
    13.30300992, 12.32848,
    11.09789888, 9.58066944
], dtype=float)

pump_coefficients = np.polyfit(flow_data, head_data, 3)

def pump_head(Q):
    """Pump head in metres for flow Q in m3/h."""
    return np.polyval(pump_coefficients, Q)

print("Pump polynomial coefficients (highest power first):")
print(pump_coefficients)


## 3. Original constant-friction-factor system curve

In [ ]:
def original_system_head(Q):
    """Original system head in metres; Q is in m3/h."""
    return STATIC_HEAD + K_ORIGINAL * np.asarray(Q)**2

def solve_operating_point(system_head_function):
    """Find pump/system intersection over the manufacturer flow range."""
    residual = lambda Q: pump_head(Q) - system_head_function(Q)
    return brentq(residual, 0.0, 96.0)

Q_original = solve_operating_point(original_system_head)
H_original = pump_head(Q_original)

print(f"Original operating flow: {Q_original:.5f} m3/h")
print(f"Original operating head: {H_original:.5f} m")
print(f"Intersection residual: {pump_head(Q_original) - original_system_head(Q_original):.3e} m")


## 4. Variable friction factor using Colebrook–White

In [ ]:
def flow_velocity(Q):
    """Mean pipe velocity in m/s for Q in m3/h."""
    Q_m3s = Q / 3600.0
    area = np.pi * D**2 / 4.0
    return Q_m3s / area

def reynolds_number(Q):
    """Reynolds number for Q in m3/h."""
    velocity = flow_velocity(Q)
    return RHO * velocity * D / MU

def colebrook_friction_factor(Re):
    """Darcy friction factor from Colebrook–White for turbulent flow."""
    if Re <= 0:
        raise ValueError("Reynolds number must be positive.")
    if Re < 4000:
        raise ValueError(
            "Colebrook–White turbulent-flow calculation is not used below Re = 4000."
        )

    def residual(f):
        return (
            1.0 / np.sqrt(f)
            + 2.0 * np.log10(
                EPSILON / (3.7 * D)
                + 2.51 / (Re * np.sqrt(f))
            )
        )

    return brentq(residual, 0.005, 0.1)

def variable_friction_factor(Q):
    if Q <= 0:
        raise ValueError("Flow rate must be positive.")
    return colebrook_friction_factor(reynolds_number(Q))

def colebrook_system_head(Q):
    """System head using flow-dependent Colebrook friction factor."""
    if Q <= 0:
        return STATIC_HEAD
    velocity = flow_velocity(Q)
    Re = reynolds_number(Q)
    if Re < 4000:
        raise ValueError("Selected flow is outside the turbulent-flow model range.")
    f = colebrook_friction_factor(Re)
    friction_head = f * (L / D) * velocity**2 / (2.0 * G)
    return STATIC_HEAD + friction_head

Q_colebrook = solve_operating_point(colebrook_system_head)
H_colebrook = pump_head(Q_colebrook)
V_colebrook = flow_velocity(Q_colebrook)
Re_colebrook = reynolds_number(Q_colebrook)
F_colebrook = variable_friction_factor(Q_colebrook)

print(f"Colebrook operating flow: {Q_colebrook:.5f} m3/h")
print(f"Colebrook operating head: {H_colebrook:.5f} m")
print(f"Velocity: {V_colebrook:.5f} m/s")
print(f"Reynolds number: {Re_colebrook:.0f}")
print(f"Darcy friction factor: {F_colebrook:.6f}")


## 5. Quantitative comparison

In [ ]:
flow_difference = Q_colebrook - Q_original
flow_difference_percent = abs(flow_difference) / Q_original * 100.0
head_difference = H_colebrook - H_original

comparison = pd.DataFrame([
    {
        "Model": "Original constant-friction-factor",
        "Operating_flow_m3h": Q_original,
        "Operating_head_m": H_original,
        "Friction_factor_at_operating_point": np.nan
    },
    {
        "Model": "Variable Colebrook friction factor",
        "Operating_flow_m3h": Q_colebrook,
        "Operating_head_m": H_colebrook,
        "Friction_factor_at_operating_point": F_colebrook
    }
])

display(comparison.round(6))
print(f"Flow difference: {flow_difference:.6f} m3/h")
print(f"Absolute flow difference: {flow_difference_percent:.5f}%")
print(f"Operating-head difference: {head_difference:.6f} m")


## 6. Friction-factor variation over the selected flow range

In [ ]:
selected_flows = np.array([10, 20, 40, 60, Q_colebrook, 80, 96], dtype=float)

profile_rows = []
for q in selected_flows:
    v = flow_velocity(q)
    Re = reynolds_number(q)
    f = colebrook_friction_factor(Re)
    profile_rows.append({
        "Flow_m3h": q,
        "Velocity_ms": v,
        "Reynolds_number": Re,
        "Darcy_friction_factor": f,
        "Colebrook_system_head_m": colebrook_system_head(q),
        "Original_system_head_m": original_system_head(q)
    })

friction_profile = pd.DataFrame(profile_rows)
display(friction_profile.round(6))


## 7. Pump and system curve comparison

In [ ]:
flow_range = np.linspace(10.0, 96.0, 200)
pump_curve = np.array([pump_head(q) for q in flow_range])
original_curve = np.array([original_system_head(q) for q in flow_range])
colebrook_curve = np.array([colebrook_system_head(q) for q in flow_range])

plt.figure(figsize=(10, 6))
plt.plot(flow_range, pump_curve, label="Pump characteristic")
plt.plot(flow_range, original_curve, "--", label="Original system curve (constant f)")
plt.plot(flow_range, colebrook_curve, label="Colebrook system curve (variable f)")
plt.scatter(Q_original, H_original, marker="o", label="Original operating point", zorder=5)
plt.scatter(Q_colebrook, H_colebrook, marker="x", s=80, label="Colebrook operating point", zorder=6)
plt.xlabel("Flow rate (m³/h)")
plt.ylabel("Head (m)")
plt.title("Pump and System Characteristic Curves")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig("Pump_System_Curve_Comparison.png", dpi=300, bbox_inches="tight")
plt.show()


## 8. Friction factor versus flow rate

In [ ]:
flow_sweep = np.linspace(10.0, 96.0, 100)
friction_sweep = np.array([variable_friction_factor(q) for q in flow_sweep])

plt.figure(figsize=(10, 5))
plt.plot(flow_sweep, friction_sweep)
plt.scatter([Q_colebrook], [F_colebrook], zorder=5, label="Colebrook operating point")
plt.xlabel("Flow rate (m³/h)")
plt.ylabel("Darcy friction factor")
plt.title("Colebrook Friction Factor Across the Flow Range")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig("Colebrook_Friction_Factor_Profile.png", dpi=300, bbox_inches="tight")
plt.show()


## 9. Export hydraulic flow-sweep results

In [ ]:
results = []

for q in flow_sweep:
    v = flow_velocity(q)
    Re = reynolds_number(q)
    f = colebrook_friction_factor(Re)
    hf = f * (L / D) * v**2 / (2.0 * G)
    h_system = STATIC_HEAD + hf
    h_pump = pump_head(q)

    results.append({
        "Flow_m3h": q,
        "Velocity_ms": v,
        "Reynolds_Number": Re,
        "Darcy_Friction_Factor": f,
        "Friction_Head_m": hf,
        "System_Head_m": h_system,
        "Pump_Head_m": h_pump,
        "Pump_System_Head_Difference_m": h_pump - h_system
    })

hydraulic_results = pd.DataFrame(results)
hydraulic_results.to_csv("Colebrook_Hydraulic_Analysis.csv", index=False)

display(hydraulic_results.head(10).round(6))
print("Saved: Colebrook_Hydraulic_Analysis.csv")


## 10. Engineering interpretation and limitations

The comparison evaluates one specific modelling choice: using a representative constant Darcy friction factor in the original system curve instead of recalculating it with the Colebrook–White equation at each flow rate.

A close match at the operating point supports the use of the representative friction factor for that operating condition. It does **not** establish that the friction factor is constant, nor does it validate every part of the pump-system model.

Interpretation should remain within these limits:
- The pump curve is a polynomial fit to manufacturer data points.
- Water properties are fixed at 25 °C.
- Only straight-pipe friction is included in this comparison; fittings, valves, entrances, exits and other minor losses are excluded.
- The original constant coefficient is compared against a Colebrook model using the stated pipe and fluid parameters.
- The analysis verifies the sensitivity of the operating point to friction-factor treatment; it does not independently validate the manufacturer's pump curve or the NPSH model.
- The Colebrook model is applied to the turbulent flow range used for the comparison.

**Conclusion:** report the numerical operating-point difference and describe whether it is practically significant for the intended engineering assessment. Avoid claiming zero error.
